In [ ]:
import numpy as np
import pandas as pd

import os
for dirname, _, _ in os.walk('/kaggle/input'):
    print(dirname)

## Preparing The Dataset

In [ ]:
os.mkdir("/kaggle/working/dataset")

In [ ]:
os.mkdir("/kaggle/working/dataset/train")
os.mkdir("/kaggle/working/dataset/test")
os.mkdir("/kaggle/working/dataset/val")

In [ ]:
dir_paths = [
    "/kaggle/input/datasets/tatheerabbas/synthetic-industrial-metal-surface-defects/industrial_defect_dataset/val",
    "/kaggle/input/datasets/tatheerabbas/synthetic-industrial-metal-surface-defects/industrial_defect_dataset/train"
]

In [ ]:
from glob import glob
from pathlib import Path

img_paths = {}

for dir_path in dir_paths:
    category_dirs = glob("{}/*".format(dir_path))

    for category_dir in category_dirs:
        category_name = Path(category_dir).stem
        if category_name not in img_paths:
            img_paths[category_name] = []
        
        image_paths = glob("{}/*".format(category_dir))
        img_paths[category_name].extend(image_paths)

In [ ]:
train_size = 0.8
test_size = 0.1

In [ ]:
from shutil import copy2
from pathlib import Path
from random import randint, shuffle

for cat_name in img_paths:
    os.mkdir("/kaggle/working/dataset/train/{}".format(cat_name))
    os.mkdir("/kaggle/working/dataset/test/{}".format(cat_name))
    os.mkdir("/kaggle/working/dataset/val/{}".format(cat_name))

    train_img_count = round(len(img_paths[cat_name]) * train_size)
    test_img_count = round(len(img_paths[cat_name]) * test_size)

    shuffle(img_paths[cat_name])

    train_img_paths = img_paths[cat_name][:train_img_count]
    test_img_paths = img_paths[cat_name][train_img_count:train_img_count+test_img_count]
    val_img_paths = img_paths[cat_name][train_img_count+test_img_count:]

    for img_path in train_img_paths:
        copy2(
            img_path,
            "/kaggle/working/dataset/train/{}/{}".format(
                cat_name,
                Path(img_path).name
            )
        )

    for img_path in test_img_paths:
        copy2(
            img_path,
            "/kaggle/working/dataset/test/{}/{}".format(
                cat_name,
                Path(img_path).name
            )
        )

    for img_path in val_img_paths:
        copy2(
            img_path,
            "/kaggle/working/dataset/val/{}/{}".format(
                cat_name,
                Path(img_path).name
            )
        )

## Training the Model

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO(model='yolo26x-cls.pt')
results = model.train(data="/kaggle/working/dataset", epochs=5, imgsz=256)

In [ ]:
yolo26x_acc1 = results.top1
print("Top-1 Accuracy:", yolo26x_acc1)